# Astrometry: parallax, proper motion, and derived peculiar-velocity precision - MAF implementation and demo


- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-11
- Context: SCOC (Survey Cadence Optimization Committee). Like notebook 04 (Kilonovae), this notebook, **05**, covers an official `rubin_sim.maf` batch group that is neither DESC (notebooks 01/01b/02/03) nor "Variables/Transients" (notebook 04): the **"Astrometry"** group, covering parallax and proper-motion precision for individual stars. It follows the same workflow as the rest of the series and is directly relevant to SCOC's stellar-population / Milky-Way-structure science case (benchmark stars, halo tracers, brown dwarfs, kinematics).
- Simulation analyzed: `/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db`
- Reproduces the official `rubin_sim.maf.batches.science_radar_batch` "Astrometry / Parallax" and "Astrometry / Proper Motion" subgroups, then adds a **derived** (not natively provided by `rubin_sim.maf`) peculiar/tangential-velocity precision, obtained by converting the proper-motion uncertainty map into a velocity uncertainty at an assumed stellar distance.


## Notebook overview

All the metrics below use a `HealpixSlicer` over the **entire survey, all visits** (the official batch uses an empty SQL constraint here, i.e. Deep-Drilling-Field visits are *included* - unlike notebooks 01-04, astrometric precision is not restricted to the WFD footprint).

1. **`ParallaxFactorStacker`** adds two columns to every visit, `ra_pi_amp` / `dec_pi_amp`: the RA/Dec offset (arcsec) that a star at 1 pc would show at that visit's epoch, due to Earth's orbital motion (the geometric parallax factor). **`DcrStacker`** similarly adds `ra_dcr_amp` / `dec_dcr_amp`, the expected image-position offset (arcsec) from differential chromatic refraction at that visit's airmass and parallactic angle.
2. **`ParallaxMetric`** combines the per-visit centroiding precision (from the visit's 5-sigma depth, seeing and an assumed fiducial-star magnitude) with the parallax-factor pattern across all visits to fit the final **parallax uncertainty** (mas) for a star of that magnitude, assuming its proper motion is either absent or perfectly known. An optional `normalize=True` mode instead reports the ratio to the *best possible* uncertainty achievable with the same total number of visits (half exactly 6 months apart) - values near 1 mean the cadence is close to astrometrically optimal for parallax.
3. **`ProperMotionMetric`** instead fits a linear position-vs-time slope through the same per-visit centroiding precisions to obtain the **proper-motion uncertainty** (mas/yr), with an analogous `normalize=True` mode comparing to the best-possible two-epoch (first/last day of the survey) cadence.
4. **`ParallaxCoverageMetric`** is a data-quality diagnostic, not an uncertainty: it checks how well the visits sample the *parallax ellipse* in phase (not just in number). Values near 1 mean the parallax motion is well sampled around its full range; values near 0 mean most visits happened to catch the star near the same point of its parallactic path (uniform sampling gives ~1 at the ecliptic poles, ~0.5 on the ecliptic itself, where the parallax motion is a straight line rather than an ellipse).
5. **`ParallaxDcrDegenMetric`** is a second diagnostic: since both parallax and DCR displace a star's apparent position by a small, periodic amount, a cadence that samples them in a correlated way (e.g. always observing at the same hour angle when the parallax factor is large) can make the two effects hard to disentangle in the astrometric fit. The metric fits both displacement patterns simultaneously and returns their correlation coefficient - near 0 is good, near +-1 means the fit will struggle to separate parallax from DCR.
6. **Derived: peculiar/tangential-velocity precision.** `rubin_sim.maf` has no dedicated "peculiar velocity" metric. But once we have a Healpix map of proper-motion uncertainty `sigma_mu` (mas/yr), the standard astrometric relation lets us convert it into a transverse-velocity uncertainty at an assumed stellar distance `d`:

$$\sigma_{v_t}\,[\mathrm{km/s}] = 4.74057 \times \sigma_\mu\,[\mathrm{arcsec/yr}] \times d\,[\mathrm{pc}]$$

(the constant 4.74057 km/s per AU/yr converts an angular rate at a given distance into a physical transverse speed). This lets us map how precisely Rubin could measure the transverse (tangential) component of a star's peculiar velocity, as a function of sky position and assumed distance - directly relevant to Milky Way kinematics and halo/stream science.


## Simulation and code references
- OpSim run analyzed: `baseline_v5.3.6_10yrs.db` (Rubin baseline v5.3.6, 10-year simulation)
- `rubin_sim.maf` source (main branch, retrieved for this notebook):
  - `rubin_sim/maf/stackers/general_stackers.py` (`ParallaxFactorStacker`, `DcrStacker`)
  - `rubin_sim/maf/metrics/calibration_metrics.py` (`ParallaxMetric`, `ProperMotionMetric`, `ParallaxCoverageMetric`, `ParallaxDcrDegenMetric`)
  - `rubin_sim/maf/metrics/area_summary_metrics.py` (`AreaSummaryMetric`, `AreaThresholdMetric`)
  - `rubin_sim/maf/batches/common.py` (`standard_summary`)
  - `rubin_sim/maf/batches/science_radar_batch.py` (official "Astrometry" batch definition, "Parallax" and "Proper Motion" subgroups, function `science_radar_batch`)
- summary.h5 / MAF outputs for standard runs: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Table of simulations: https://usdf-maf.slac.stanford.edu/


## 1. Imports

In [ ]:
import os
import inspect
from os.path import splitext, basename

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.metric_bundles as mb
from rubin_sim.maf.batches.common import standard_summary
from rubin_sim.maf.stackers import ParallaxFactorStacker, DcrStacker
from rubin_sim.maf.metrics.calibration_metrics import (
    ParallaxMetric,
    ProperMotionMetric,
    ParallaxCoverageMetric,
    ParallaxDcrDegenMetric,
)

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

Same OpSim file and output-directory convention as the rest of the series, with a dedicated `NB_TAG`.

In [ ]:
opsim_fname = "/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db"
assert os.path.isfile(opsim_fname), f"OpSim database not found: {opsim_fname}"

run_name = splitext(basename(opsim_fname))[0]
print("run_name:", run_name)

In [ ]:
NB_TAG = "ASTROMETRY"
data_dir = f"data_05_{NB_TAG}"
figs_dir = f"figs_05_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

In [ ]:
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. The MAF classes used for astrometric precision

In [ ]:
print(inspect.getdoc(ParallaxFactorStacker))
print("-" * 80)
print(inspect.getdoc(DcrStacker))
print("-" * 80)
print(inspect.getdoc(ParallaxMetric))
print("-" * 80)
print(inspect.getdoc(ProperMotionMetric))

In [ ]:
print(inspect.getdoc(ParallaxCoverageMetric))
print("-" * 80)
print(inspect.getdoc(ParallaxDcrDegenMetric))

## 4. Configuration (matching the official `science_radar_batch` "Astrometry" subgroups)

Two fiducial-star magnitudes are used for each family, exactly as in the official batch: a brighter and a fainter benchmark star. The SQL constraint is left **empty** (all visits, DDF included), also matching the official batch - astrometric precision benefits from every repeated visit anywhere on sky.

In [ ]:
nside = 64
sqlconstraint = ""  # all visits, including DDF - matches the official Astrometry batch group

rmags_para = [22.4, 24.0]  # fiducial star magnitudes for the parallax metrics
rmags_pm = [20.5, 24.0]  # fiducial star magnitudes for the proper-motion metrics
good_parallax_limit = 11.5  # mas; the official batch's "good" parallax-uncertainty threshold

parallax_stacker = ParallaxFactorStacker()
dcr_stacker = DcrStacker()

healpixslicer = slicers.HealpixSlicer(nside=nside, use_cache=False)
pix_area = hp.nside2pixarea(nside, degrees=True)
print(f"nside={nside} -> pixel area = {pix_area:.4f} deg^2")

## 5. Running the parallax metrics

For each fiducial magnitude, we run the plain `ParallaxMetric` (uncertainty in mas) and the `normalize=True` version (ratio to the best-possible scheduling), plus the two diagnostics (`ParallaxCoverageMetric`, `ParallaxDcrDegenMetric`). The summary statistics reproduce the official batch: `standard_summary()` (mean/median/rms/min/max/outlier counts) plus, for the plain uncertainty, `AreaSummaryMetric` (median over the best 18,000 deg^2 - a stand-in for "the WFD value"), `AreaThresholdMetric` (area better than the `good_parallax_limit`), and the 95th-percentile uncertainty.

In [ ]:
def parallax_summary():
    summary = [
        metrics.AreaSummaryMetric(
            area=18000,
            reduce_func=np.median,
            decreasing=False,
            metric_name="Median Parallax Uncert (18k)",
        ),
        metrics.AreaThresholdMetric(
            upper_threshold=good_parallax_limit,
            metric_name=f"Area better than {good_parallax_limit:.1f} mas uncertainty",
        ),
        metrics.PercentileMetric(percentile=95, metric_name="95th Percentile Parallax Uncert"),
    ]
    summary.extend(standard_summary())
    return summary


parallax_bundles = {}
parallax_norm_bundles = {}
coverage_bundles = {}
dcr_degen_bundles = {}

for rmag in rmags_para:
    parallax_bundles[rmag] = mb.MetricBundle(
        ParallaxMetric(metric_name=f"Parallax Uncert @ {rmag}", rmag=rmag, normalize=False),
        healpixslicer,
        sqlconstraint,
        stacker_list=[parallax_stacker],
        run_name=run_name,
        summary_metrics=parallax_summary(),
    )
    parallax_norm_bundles[rmag] = mb.MetricBundle(
        ParallaxMetric(metric_name=f"Normalized Parallax Uncert @ {rmag}", rmag=rmag, normalize=True),
        healpixslicer,
        sqlconstraint,
        stacker_list=[parallax_stacker],
        run_name=run_name,
        summary_metrics=standard_summary(),
    )
    coverage_bundles[rmag] = mb.MetricBundle(
        ParallaxCoverageMetric(metric_name=f"Parallax Coverage @ {rmag}", rmag=rmag),
        healpixslicer,
        sqlconstraint,
        stacker_list=[parallax_stacker],
        run_name=run_name,
        summary_metrics=standard_summary(),
    )
    dcr_degen_bundles[rmag] = mb.MetricBundle(
        ParallaxDcrDegenMetric(metric_name=f"Parallax-DCR degeneracy @ {rmag}", rmag=rmag),
        healpixslicer,
        sqlconstraint,
        stacker_list=[dcr_stacker, parallax_stacker],
        run_name=run_name,
        summary_metrics=standard_summary(),
    )

all_parallax_bundles = (
    list(parallax_bundles.values())
    + list(parallax_norm_bundles.values())
    + list(coverage_bundles.values())
    + list(dcr_degen_bundles.values())
)
bd = mb.make_bundles_dict_from_list(all_parallax_bundles)
bgroup = mb.MetricBundleGroup(bd, opsim_fname, out_dir=data_dir, results_db=resultsDb)
bgroup.run_all()
print("Done.")

## 6. Running the proper-motion metrics

In [ ]:
def pm_summary():
    summary = [
        metrics.AreaSummaryMetric(
            area=18000,
            reduce_func=np.median,
            decreasing=False,
            metric_name="Median Proper Motion Uncert (18k)",
        ),
        metrics.PercentileMetric(percentile=95, metric_name="95th Percentile Proper Motion Uncert"),
    ]
    summary.extend(standard_summary())
    return summary


pm_bundles = {}
pm_norm_bundles = {}
for rmag in rmags_pm:
    pm_bundles[rmag] = mb.MetricBundle(
        ProperMotionMetric(metric_name=f"Proper Motion Uncert @ {rmag}", rmag=rmag, normalize=False),
        healpixslicer,
        sqlconstraint,
        run_name=run_name,
        summary_metrics=pm_summary(),
    )
    pm_norm_bundles[rmag] = mb.MetricBundle(
        ProperMotionMetric(
            metric_name=f"Normalized Proper Motion Uncert @ {rmag}", rmag=rmag, normalize=True
        ),
        healpixslicer,
        sqlconstraint,
        run_name=run_name,
        summary_metrics=standard_summary(),
    )

all_pm_bundles = list(pm_bundles.values()) + list(pm_norm_bundles.values())
bd_pm = mb.make_bundles_dict_from_list(all_pm_bundles)
bgroup_pm = mb.MetricBundleGroup(bd_pm, opsim_fname, out_dir=data_dir, results_db=resultsDb)
bgroup_pm.run_all()
print("Done.")

## 7. Results table

In [ ]:
rows = []
for rmag in rmags_para:
    sv = parallax_bundles[rmag].summary_values
    rows.append(
        {
            "quantity": f"Parallax Uncert @ r={rmag}",
            "units": "mas",
            "median (18k deg2)": sv.get("Median Parallax Uncert (18k)"),
            "95th percentile": sv.get("95th Percentile Parallax Uncert"),
            "median (all sky)": sv.get("Median"),
        }
    )
for rmag in rmags_pm:
    sv = pm_bundles[rmag].summary_values
    rows.append(
        {
            "quantity": f"Proper Motion Uncert @ r={rmag}",
            "units": "mas/yr",
            "median (18k deg2)": sv.get("Median Proper Motion Uncert (18k)"),
            "95th percentile": sv.get("95th Percentile Proper Motion Uncert"),
            "median (all sky)": sv.get("Median"),
        }
    )
for rmag in rmags_para:
    sv = coverage_bundles[rmag].summary_values
    rows.append(
        {
            "quantity": f"Parallax Coverage @ r={rmag} (0 bad, ~1 good)",
            "units": "ratio",
            "median (18k deg2)": np.nan,
            "95th percentile": np.nan,
            "median (all sky)": sv.get("Median"),
        }
    )
for rmag in rmags_para:
    sv = dcr_degen_bundles[rmag].summary_values
    rows.append(
        {
            "quantity": f"Parallax-DCR degeneracy @ r={rmag} (0 good, +-1 bad)",
            "units": "correlation",
            "median (18k deg2)": np.nan,
            "95th percentile": np.nan,
            "median (all sky)": sv.get("Median"),
        }
    )
results_df = pd.DataFrame(rows).set_index("quantity")
results_df

In [ ]:
results_csv = os.path.join(data_dir, f"{run_name}_astrometry_summary.csv")
results_df.to_csv(results_csv)
print("Saved:", results_csv)

## 8. Healpix maps and histograms

Native MAF `HealpixSkyMap` + `HealpixHistogram` plots (`bundle.plot()`) for the bright- and faint-star parallax and proper-motion uncertainty maps, and for the two diagnostics.

In [ ]:
def save_bundle_plots(bundle, tag, figs_dir):
    made_plots = bundle.plot(savefig=False)
    saved = []
    for plot_type, fig in made_plots.items():
        if fig is None:
            continue
        base = os.path.join(figs_dir, f"{tag}_{plot_type}")
        fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
        fig.savefig(base + ".pdf", bbox_inches="tight")
        saved.append(base)
        plt.close(fig)
    return saved

In [ ]:
for rmag, bundle in parallax_bundles.items():
    print(f"--- Parallax uncertainty, r={rmag} ---")
    saved = save_bundle_plots(bundle, f"{run_name}_ParallaxUncert_r{rmag}", figs_dir)
    for s in saved:
        print("  saved:", s + ".png/.pdf")
_ = parallax_bundles[rmags_para[0]].plot(savefig=False)
plt.show()

In [ ]:
for rmag, bundle in pm_bundles.items():
    print(f"--- Proper motion uncertainty, r={rmag} ---")
    saved = save_bundle_plots(bundle, f"{run_name}_ProperMotionUncert_r{rmag}", figs_dir)
    for s in saved:
        print("  saved:", s + ".png/.pdf")
_ = pm_bundles[rmags_pm[0]].plot(savefig=False)
plt.show()

In [ ]:
print(f"--- Parallax coverage, r={rmags_para[0]} ---")
saved = save_bundle_plots(
    coverage_bundles[rmags_para[0]], f"{run_name}_ParallaxCoverage_r{rmags_para[0]}", figs_dir
)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = coverage_bundles[rmags_para[0]].plot(savefig=False)
plt.show()

print(f"--- Parallax-DCR degeneracy, r={rmags_para[0]} ---")
saved = save_bundle_plots(
    dcr_degen_bundles[rmags_para[0]], f"{run_name}_ParallaxDcrDegen_r{rmags_para[0]}", figs_dir
)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = dcr_degen_bundles[rmags_para[0]].plot(savefig=False)
plt.show()

## 9. Derived: peculiar (tangential) velocity precision

Converting the `r=20.5` proper-motion uncertainty Healpix map into a transverse-velocity uncertainty at two illustrative distances: 1 kpc (a typical nearby disk/thick-disk tracer) and 8 kpc (comparable to the distance to the Galactic center, typical of many halo tracers). This is a plain unit conversion of the existing map, done here in the notebook rather than inside `rubin_sim.maf`.

In [ ]:
KM_S_PER_AU_YR = 4.74057  # km/s per (arcsec/yr at 1 pc); i.e. per AU/yr


def pm_uncert_to_vt_uncert(pm_mas_yr, distance_pc):
    """Convert a proper-motion uncertainty (mas/yr) to a transverse-velocity
    uncertainty (km/s) at the given distance (pc)."""
    pm_arcsec_yr = pm_mas_yr / 1000.0
    return KM_S_PER_AU_YR * pm_arcsec_yr * distance_pc


pm_bright_bundle = pm_bundles[rmags_pm[0]]
pm_map_mas_yr = pm_bright_bundle.metric_values  # masked array, mas/yr, one value per Healpix pixel

distances_pc = {"1 kpc": 1000.0, "8 kpc": 8000.0}
vt_maps = {label: pm_uncert_to_vt_uncert(pm_map_mas_yr, d) for label, d in distances_pc.items()}

for label, vt_map in vt_maps.items():
    good = vt_map.compressed()
    print(
        f"Tangential-velocity uncertainty at {label} (r={rmags_pm[0]} star): "
        f"median = {np.median(good):.2f} km/s, 95th percentile = {np.percentile(good, 95):.2f} km/s"
    )

In [ ]:
def plot_and_save_vt_map(vt_map, label, tag, figs_dir):
    fig = plt.figure(figsize=(8, 5))
    hp.mollview(
        vt_map.filled(hp.UNSEEN),
        fig=fig.number,
        title=f"Tangential-velocity uncertainty at {label} (r={rmags_pm[0]} star)",
        unit="km/s",
        cmap="viridis",
    )
    base = os.path.join(figs_dir, f"{tag}_vt_uncert_{label.replace(' ', '')}")
    fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
    fig.savefig(base + ".pdf", bbox_inches="tight")
    print("Saved:", base + ".png/.pdf")
    plt.show()


for label, vt_map in vt_maps.items():
    plot_and_save_vt_map(vt_map, label, run_name, figs_dir)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
for label, vt_map in vt_maps.items():
    ax.hist(vt_map.compressed(), bins=60, histtype="step", lw=1.5, label=label, density=True)
ax.set_xlabel("Tangential-velocity uncertainty [km/s]")
ax.set_ylabel("Normalized pixel count")
ax.set_title(f"Peculiar-velocity precision (r={rmags_pm[0]} star) - {run_name}")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_vt_uncert_histograms")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 10. Caveats

- `ParallaxMetric` and `ProperMotionMetric` both assume a single fiducial-magnitude, flat-SED star with no astrophysical noise floor (no blending/crowding, no reference-frame or calibration systematics) and, for `ParallaxMetric`, either zero proper motion or a perfectly independently known one (and vice versa for `ProperMotionMetric`) - real joint parallax+proper-motion fits are somewhat worse than either metric alone suggests.
- The `sqlconstraint=""` used here (matching the official batch) includes **all** visits, DDF included - unlike notebooks 01-04, which explicitly excluded DDF visits.
- The peculiar/tangential-velocity conversion in Section 9 propagates only the **proper-motion uncertainty**, not any distance uncertainty (a real parallax-derived distance has its own error, which would need to be propagated too for a full peculiar-velocity error budget), and it only captures the **tangential** component - the radial (line-of-sight) velocity component requires spectroscopy from another facility, not Rubin astrometry.
- `ParallaxCoverageMetric` and `ParallaxDcrDegenMetric` are cadence-quality diagnostics, not uncertainties in the usual sense; their "good"/"bad" thresholds (documented in their docstrings above) are heuristic, calibrated on Monte Carlo experience rather than a hard analytic criterion.


## References
- Ivezic, Z. et al. 2019, ApJ 873, 111, "LSST: From Science Drivers to Reference Design and Anticipated Data Products" - main Rubin/LSST survey design paper, including the astrometric performance requirements this batch's fiducial magnitudes are drawn from.
- LSST Science Collaboration 2009, "LSST Science Book", arXiv:0912.0201 - Milky Way structure and stellar kinematics science case (Chapter on Galactic Structure).
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/metrics/calibration_metrics.py`, `rubin_sim/maf/stackers/general_stackers.py`, `rubin_sim/maf/batches/science_radar_batch.py`)
